In [1]:
import jax
import jax.numpy as jnp
import matplotlib.image as mpimg
import matplotlib.pyplot
import ott
from ott.geometry import geometry, pointcloud
from ott.solvers import linear

In [2]:
key = jax.random.key(0)

In [3]:
def SR_proj(K, gamma, tau, a, b, delta, r):
    u = jnp.ones((n))
    v = jnp.ones((r))
    G = 0
    while G<delta:
        u_k = u
        v_k = v
        u = a/(K@v)
        v = (b/(jnp.transpose(K)@u))**(tau/(tau +1/gamma))
        G = max(jnp.linalg.norm(jnp.log(u_k/u)), jnp.linalg.norm(jnp.log(v_k/v)))
    return jnp.diag(u)@K@jnp.diag(v)

In [4]:
def sinkhorn(K, a, b, delta=1e-2):
    u, v = linear.solve(geometry.Geometry(kernel_matrix=K, epsilon=delta), a=a, b=b).potentials
    return jnp.diag(u)@K@jnp.diag(v)

In [5]:
def init(a, b, gQ, gR):
    keys = jax.random.split(key, 3)
    n = len(a)
    m = len(b)
    r = len(gQ)
    CQ = jax.random.uniform(keys[0], (n,r))
    CR = jax.random.uniform(keys[1], (m,r))
    CT = jax.random.uniform(keys[2], (r,r))
    KQ = jnp.exp(CQ)
    KR = jnp.exp(CR)
    KT = jnp.exp(CT)
    Q = sinkhorn(KQ, a, gQ)
    R = sinkhorn(KR, b, gR)
    T = sinkhorn(KT, jnp.transpose(Q)@jnp.ones((n)), jnp.transpose(R)@jnp.ones((m)))
    return Q, R, T

In [17]:
def Balanced_FRLC(C, r, a, b, tau, gamma, delta, eps):
    n, m = jnp.shape(C)
    gQ = (1/r)*jnp.ones((r))
    gR = (1/r)*jnp.ones((r))
    Q, R, T = init(a, b, gQ, gR)
    X = jnp.diag(1/(jnp.transpose(Q)@jnp.ones((n,r))))@T@jnp.diag(1/(jnp.transpose(R)@jnp.ones((m))))
    Delta = float('inf')
    while Delta > eps:
        temp = C@R@jnp.transpose(X)
        print(jnp.shape(jnp.transpose(temp)@Q@jnp.diag(1/gQ)))
        GradQ = temp - jnp.ones((n,1))@jnp.reshape(jnp.diag(jnp.transpose(temp)@Q@jnp.diag(1/gQ)), (1,r))
        temp = jnp.transpose(C)@Q@X
        GradR = temp - jnp.ones((m,1))@jnp.reshape(jnp.diag(jnp.diag(1/gR)@jnp.transpose(R)@temp), (1,r))
        gamma_k = gamma/max(jnp.linalg.norm(GradQ, ord='inf'), jnp.linalg.norm(GradR, ord='inf'))
        KQ = jnp.multiply(Q,jnp.exp(-gamma_k*GradQ))
        KR = jnp.multiply(R,jnp.exp(-gamma_k*GradR))
        Q_new = SR_proj(KQ, gamma_k, tau, a, jnp.transpose(Q)@jnp.ones((n)), delta, r)
        R_new = SR_proj(KR, gamma_k, tau, b, jnp.transpose(R)@jnp.ones((m)), delta, r)
        gQ = jnp.transpose(Q_new)@jnp.ones((n))
        gR = jnp.transpose(R_new)@jnp.ones((m))
        GradT = jnp.diag(1/gQ)@jnp.transpose(Q_new)@C@R_new@jnp.diag(1/gR)
        gamma_t = gamma/jnp.linalg.norm(GradT, ord='inf')
        KT = jnp.multiply(T, jnp.exp(-gamma_t*GradT))
        T_new = sinkhorn(KT, gR, gQ, delta)
        X = jnp.diag(1/gQ)@T@jnp.diag(1/gR)
        Delta = (1/gamma_k**2) * (jnp.linalg.norm(Q_new - Q) + jnp.linalg.norm(R_new - R) + jnp.linalg.norm(T_new - T)) 
        Q = Q_new
        R = R_new
        T = T_new
    return Q@X@jnp.transpose(R)


In [10]:
#Create data
n = 10
m = 15
r = 5

keys = jax.random.split(key, 4)
x = jax.random.normal(keys[0], (n,r)) + 1
y = jax.random.uniform(keys[1], (m,r))
a = jax.random.uniform(keys[2], (n))
b = jax.random.uniform(keys[3], (m))
a = a/jnp.sum(a)
b = b/jnp.sum(b)

In [11]:
gamma = 1e-2
delta = 1e-2
tau = 1e-2
eps = 1e-2

In [12]:
C = pointcloud.PointCloud(x,y).cost_matrix

In [13]:
n, m = jnp.shape(C)
gQ = (1/r)*jnp.ones((r))
gR = (1/r)*jnp.ones((r))
keys = jax.random.split(key, 3)
n = len(a)
m = len(b)
r = len(gQ)
CQ = jax.random.uniform(keys[0], (n,r))
CR = jax.random.uniform(keys[1], (m,r))
CT = jax.random.uniform(keys[2], (r,r))
KQ = jnp.exp(CQ)
KR = jnp.exp(CR)
KT = jnp.exp(CT)
Q = sinkhorn(KQ, a, gQ)
jnp.shape(Q)

(10, 5)

In [18]:
P = Balanced_FRLC(C, r, a, b, tau, gamma, delta, eps)

(5,)


TypeError: cannot reshape array of shape (5, 5) (size 25) into shape (1, 5) (size 5)